# Westwood Curbside Parking: Longitudinal Overstay Analysis (2021–2023)

This notebook covers:
1. Load & merge all monthly sensor files
2. Build parking sessions (OCCUPIED → VACANT pairs)
3. Correct overstay detection (enforcement-window-aware)
4. Feature engineering (UCLA quarter, weekday/weekend, time-of-day)
5. Save processed dataset
6. Descriptive statistics
7. Temporal analysis (trends by year, quarter, weekday)
8. Spatial / hotspot analysis
9. ML — overstay prediction

## 1. Setup

In [ ]:
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)

DATA_DIR   = '/Users/tatsrisan/Desktop/Research/Parking/'
SENSOR_CSV = DATA_DIR + 'westwood_sensors.csv'   # per-spot enforcement windows

# Westwood bounding box (same as previous notebooks)
WW_MIN_LAT, WW_MAX_LAT = 34.046647, 34.082840
WW_MIN_LON, WW_MAX_LON = -118.466525, -118.414076

# UCLA quarter mapping
QUARTER_MAP = {
    'January': 'Winter', 'February': 'Winter', 'March': 'Winter',
    'April': 'Spring',   'May': 'Spring',      'June': 'Spring',
    'July': 'Summer',    'August': 'Summer',   'September': 'Summer',
    'October': 'Fall',   'November': 'Fall',   'December': 'Fall',
}

DAY_COL = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']  # maps to dayofweek 0-6

## 2. Helper functions

In [ ]:
def parse_location(csv_path):
    """Load LADOT inventory and extract lat/lon."""
    loc = pd.read_csv(DATA_DIR + 'LADOT_Metered_Parking_Inventory___Policies.csv')
    loc[['Latitude', 'Longitude']] = loc['LatLng'].apply(
        lambda s: pd.Series(eval(s) if pd.notna(s) else (None, None))
    )
    return loc[['SpaceID', 'BlockFace', 'MeterType', 'RateType',
                'RateRange', 'MeteredTimeLimit', 'Latitude', 'Longitude']]


def parse_enforcement_window(window_str):
    """'8.00-20.00' → (8.0, 20.0). '8.00-24.00' → (8.0, 24.0). NaN → None."""
    if pd.isna(window_str):
        return None
    try:
        start, end = window_str.split('-')
        return float(start), float(end)
    except Exception:
        return None


def parse_time_limit(limit_str):
    """Convert MeteredTimeLimit string to total minutes.
    '2HR' → 120, '30MIN' → 30, '4HR-30MIN' → 270 (compound limit, take the larger).
    """
    if pd.isna(limit_str):
        return np.nan
    s = limit_str.strip().upper()
    # Compound: '4HR-30MIN' or '2HR-4HR'
    if s.count('-') == 1 and ('HR' in s and s.index('-') > 0):
        parts = s.split('-')
        minutes = [parse_time_limit(p) for p in parts]
        minutes = [m for m in minutes if not np.isnan(m)]
        return max(minutes) if minutes else np.nan
    if 'HR' in s:
        return float(s.replace('HR', '').strip()) * 60
    if 'MIN' in s:
        return float(s.replace('MIN', '').strip())
    return np.nan


def is_enforced(session_start_dt, day_of_week, enforcement_windows):
    """
    Returns True if a session starting at session_start_dt is within
    the enforcement window for that day.
    enforcement_windows: dict {dayofweek (0=Mon) → (start_h, end_h)}
    """
    window = enforcement_windows.get(day_of_week)
    if window is None:
        return False
    start_h, end_h = window
    hour = session_start_dt.hour + session_start_dt.minute / 60.0
    return start_h <= hour < end_h


# Precompute enforcement windows per SpaceID per day
def build_enforcement_table(sensor_df):
    """Returns dict: {SpaceID → {0:Mon, 1:Tue, ..., 6:Sun} → (start_h, end_h) or None}"""
    table = {}
    for _, row in sensor_df.iterrows():
        sid = row['SpaceID']
        table[sid] = {}
        for i, col in enumerate(DAY_COL):
            table[sid][i] = parse_enforcement_window(row.get(col))
    return table

## 3. Load all monthly files → filter Westwood → build sessions

In [ ]:
location = parse_location(DATA_DIR)

sensor_info = pd.read_csv(SENSOR_CSV)
# Keep only enforcement-window columns + SpaceID
sensor_meta = sensor_info[['SpaceID'] + DAY_COL].copy()

# Merge location with enforcement windows
location = location.merge(sensor_meta, on='SpaceID', how='left')

# Get Westwood SpaceIDs (those within bounding box)
ww_spaces = set(
    location[
        (location['Latitude'] >= WW_MIN_LAT) & (location['Latitude'] <= WW_MAX_LAT) &
        (location['Longitude'] >= WW_MIN_LON) & (location['Longitude'] <= WW_MAX_LON)
    ]['SpaceID']
)
print(f'Westwood spots: {len(ww_spaces)}')

# Pre-build enforcement table
ww_location = location[location['SpaceID'].isin(ww_spaces)].copy()
ww_location['TimeLimitMinutes'] = ww_location['MeteredTimeLimit'].apply(parse_time_limit)
enforcement_table = build_enforcement_table(
    ww_location.drop_duplicates('SpaceID').set_index('SpaceID')
    .reset_index()[['SpaceID'] + DAY_COL]
)
print('Enforcement table built.')

In [ ]:
def load_month(filepath, ww_spaces, location_df):
    """Load one monthly CSV, filter Westwood, parse datetime."""
    df = pd.read_csv(filepath, usecols=['SpaceID', 'EventTime_Local', 'OccupancyState'])
    df = df[df['SpaceID'].isin(ww_spaces)].copy()
    if df.empty:
        return None
    df['DateTime'] = pd.to_datetime(df['EventTime_Local'], infer_datetime_format=True)
    df.drop(columns=['EventTime_Local'], inplace=True)
    df = df.merge(
        location_df[['SpaceID', 'BlockFace', 'MeteredTimeLimit', 'TimeLimitMinutes',
                     'RateType', 'Latitude', 'Longitude'] + DAY_COL],
        on='SpaceID', how='left'
    )
    return df


def build_sessions(df):
    """
    Vectorised session builder.
    A session = consecutive OCCUPIED → VACANT for the same SpaceID.
    Returns one row per valid session (at the OCCUPIED event).
    """
    df = df.sort_values(['SpaceID', 'DateTime']).reset_index(drop=True)
    grp = df.groupby('SpaceID')
    df['next_state'] = grp['OccupancyState'].shift(-1)
    df['next_time']  = grp['DateTime'].shift(-1)

    sessions = df[
        (df['OccupancyState'] == 'OCCUPIED') & (df['next_state'] == 'VACANT')
    ].copy()
    sessions['SessionStart']    = sessions['DateTime']
    sessions['SessionEnd']      = sessions['next_time']
    sessions['OccupancyMin']    = (sessions['SessionEnd'] - sessions['SessionStart']).dt.total_seconds() / 60
    return sessions.drop(columns=['OccupancyState', 'next_state', 'next_time', 'DateTime'])


def compute_overstay(sessions, enforcement_table):
    """
    Adds columns:
      DayOfWeek      : 0=Mon … 6=Sun
      StartHour      : float hour of session start
      IsDuringEnforcement : bool
      IsOverstay     : bool (only True if enforcement active)
      OverstayMin    : minutes over limit (0 if no overstay)
    """
    sessions = sessions.copy()
    sessions['DayOfWeek'] = sessions['SessionStart'].dt.dayofweek  # 0=Mon
    sessions['StartHour'] = sessions['SessionStart'].dt.hour + sessions['SessionStart'].dt.minute / 60

    def check_enforcement(row):
        sid = row['SpaceID']
        window = enforcement_table.get(sid, {}).get(row['DayOfWeek'])
        if window is None:
            return False
        start_h, end_h = window
        return start_h <= row['StartHour'] < end_h

    sessions['IsDuringEnforcement'] = sessions.apply(check_enforcement, axis=1)
    sessions['IsOverstay'] = (
        sessions['IsDuringEnforcement'] &
        (sessions['OccupancyMin'] > sessions['TimeLimitMinutes'])
    )
    sessions['OverstayMin'] = np.where(
        sessions['IsOverstay'],
        sessions['OccupancyMin'] - sessions['TimeLimitMinutes'],
        0
    )
    return sessions

In [ ]:
all_files = sorted(glob.glob(DATA_DIR + 'Sensor_Transactions_202[1-9]*.csv'))
print(f'Files to process: {len(all_files)}')  # should be 36 (2021–2023)

session_chunks = []
for fp in all_files:
    label = fp.split('/')[-1]
    raw = load_month(fp, ww_spaces, ww_location)
    if raw is None:
        print(f'  {label}: no Westwood data, skipped')
        continue
    sess = build_sessions(raw)
    sess = compute_overstay(sess, enforcement_table)
    session_chunks.append(sess)
    print(f'  {label}: {len(sess):,} sessions, {sess["IsOverstay"].sum():,} overstays')

sessions = pd.concat(session_chunks, ignore_index=True)
print(f'\nTotal sessions: {len(sessions):,}')

## 4. Feature engineering

In [ ]:
sessions['Year']          = sessions['SessionStart'].dt.year
sessions['Month']         = sessions['SessionStart'].dt.month
sessions['MonthName']     = sessions['SessionStart'].dt.month_name()
sessions['Quarter']       = sessions['MonthName'].map(QUARTER_MAP)
sessions['IsWeekend']     = sessions['DayOfWeek'].isin([5, 6])  # Sat=5, Sun=6
sessions['DayName']       = sessions['SessionStart'].dt.day_name()

# Time of day bins
bins  = [0, 8, 12, 17, 20, 24]
labels = ['Overnight (0–8)', 'Morning (8–12)', 'Afternoon (12–17)', 'Evening (17–20)', 'Night (20–24)']
sessions['TimeOfDay'] = pd.cut(sessions['StartHour'], bins=bins, labels=labels, right=False)

# Year-Quarter label for time-series plots
sessions['YearQuarter'] = sessions['Year'].astype(str) + '-' + sessions['Quarter']

sessions.info()

## 5. Save processed dataset

In [ ]:
out_path = DATA_DIR + 'westwood_sessions_all.parquet'
sessions.to_parquet(out_path, index=False)
print(f'Saved → {out_path}')

# Quick reload check
# sessions = pd.read_parquet(out_path)

---
## 6. Descriptive Statistics

Covers all sessions during enforcement hours (the denominator that matters).

In [ ]:
enforced = sessions[sessions['IsDuringEnforcement']].copy()

total_sessions  = len(enforced)
total_overstays = enforced['IsOverstay'].sum()
overstay_rate   = total_overstays / total_sessions * 100

print('=== Overall (enforcement hours only) ===')
print(f'Total sessions       : {total_sessions:,}')
print(f'Overstay sessions    : {total_overstays:,} ({overstay_rate:.1f}%)')
print(f'Median stay (min)    : {enforced["OccupancyMin"].median():.1f}')
print(f'Mean stay (min)      : {enforced["OccupancyMin"].mean():.1f}')
print()
print('--- Overstaying sessions only ---')
overstay_df = enforced[enforced['IsOverstay']]
print(f'Median overstay (min): {overstay_df["OverstayMin"].median():.1f}')
print(f'Mean overstay (min)  : {overstay_df["OverstayMin"].mean():.1f}')
print(f'Max overstay (min)   : {overstay_df["OverstayMin"].max():.1f}')

In [ ]:
# Session duration distribution (enforced, capped at 8h for readability)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cap = 480  # 8 hours in minutes
enforced['OccupancyMin'].clip(upper=cap).hist(
    bins=60, ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Session duration (enforced hours, capped 8h)')
axes[0].set_xlabel('Minutes')
axes[0].set_ylabel('Sessions')

overstay_df['OverstayMin'].clip(upper=cap).hist(
    bins=60, ax=axes[1], color='tomato', edgecolor='white'
)
axes[1].set_title('Overstay duration (minutes over limit, capped 8h)')
axes[1].set_xlabel('Minutes over limit')
axes[1].set_ylabel('Sessions')

plt.tight_layout()
plt.savefig(DATA_DIR + 'fig_duration_dist.png', dpi=150)
plt.show()

In [ ]:
# Overstay rate by overstay-duration bucket
bins_ov   = [0, 15, 30, 60, 120, 240, np.inf]
lab_ov    = ['<15min', '15–30', '30–60', '1–2h', '2–4h', '>4h']
overstay_df['OverstayBucket'] = pd.cut(overstay_df['OverstayMin'], bins=bins_ov, labels=lab_ov)

bucket_counts = overstay_df['OverstayBucket'].value_counts().sort_index()
print('Overstay duration distribution:')
print(bucket_counts.to_string())

---
## 7. Temporal Analysis

In [ ]:
# 7a. Annual overstay rate
annual = enforced.groupby('Year').agg(
    Sessions=('IsOverstay', 'count'),
    Overstays=('IsOverstay', 'sum'),
    MeanOccMin=('OccupancyMin', 'mean'),
    MeanOverstayMin=('OverstayMin', lambda x: x[x > 0].mean())
).assign(OverstayRate=lambda d: d['Overstays'] / d['Sessions'] * 100)

print('Annual summary:')
print(annual.round(2).to_string())

### 7a-ii. COVID-19 Recovery Context
2021 is a partial-recovery year (LA restrictions lifted ~June 2021). Lower session counts
reflect reduced activity; the overstay *rate* in 2021 may differ from 2022–2023 as
drivers returned gradually. We annotate the monthly trend accordingly.

In [ ]:
# COVID recovery: monthly session volume + overstay rate with period annotations
monthly = enforced.groupby(['Year', 'Month']).agg(
    Sessions=('IsOverstay', 'count'),
    Overstays=('IsOverstay', 'sum'),
).assign(OverstayRate=lambda d: d['Overstays'] / d['Sessions'] * 100).reset_index()
monthly['Date'] = pd.to_datetime(monthly[['Year', 'Month']].assign(day=1))
monthly = monthly.sort_values('Date')

# LA indoor restrictions lifted June 15 2021; masks lifted Feb 16 2022
COVID_EVENTS = {
    '2021-06-15': 'LA restrictions\nlifted',
    '2022-02-16': 'Mask mandate\nlifted',
}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Top: session volume
ax1.bar(monthly['Date'], monthly['Sessions'], width=25, color='steelblue', alpha=0.8, label='Enforced sessions')
ax1.set_ylabel('Sessions per month')
ax1.set_title('Westwood Curbside Parking — COVID Recovery Context (2021–2023)')

# Bottom: overstay rate
ax2.plot(monthly['Date'], monthly['OverstayRate'], marker='o', color='tomato', linewidth=1.8, label='Overstay rate')
ax2.set_ylabel('Overstay Rate (%)')
ax2.set_xlabel('')

# Shade 2021 as COVID recovery year
import matplotlib.dates as mdates
recovery_start = pd.Timestamp('2021-01-01')
recovery_end   = pd.Timestamp('2021-12-31')
for ax in (ax1, ax2):
    ax.axvspan(recovery_start, recovery_end, color='gold', alpha=0.18, label='2021 Recovery year')
    for date_str, label in COVID_EVENTS.items():
        dt = pd.Timestamp(date_str)
        ax.axvline(dt, color='grey', linestyle='--', linewidth=1)
        ax.text(dt, ax.get_ylim()[1] * 0.92, label, fontsize=7.5,
                ha='center', va='top', color='grey',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6))

ax1.legend(loc='upper left', fontsize=9)
ax2.legend(loc='upper left', fontsize=9)

plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(DATA_DIR + 'fig_covid_recovery.png', dpi=150)
plt.show()

# Print year-over-year % change in sessions and overstay rate
print('Year-over-year change:')
yoy = annual[['Sessions', 'Overstays', 'OverstayRate']].pct_change() * 100
yoy.columns = ['Sessions_chg%', 'Overstays_chg%', 'OverstayRate_chg%']
print(pd.concat([annual, yoy], axis=1).round(2).to_string())

In [ ]:
# 7b. Overstay rate by UCLA Quarter × Year
quarter_order = ['Winter', 'Spring', 'Summer', 'Fall']

qy = enforced.groupby(['Year', 'Quarter']).agg(
    Sessions=('IsOverstay', 'count'),
    Overstays=('IsOverstay', 'sum'),
).assign(OverstayRate=lambda d: d['Overstays'] / d['Sessions'] * 100).reset_index()

pivot_rate = qy.pivot(index='Quarter', columns='Year', values='OverstayRate').loc[quarter_order]
pivot_sess = qy.pivot(index='Quarter', columns='Year', values='Sessions').loc[quarter_order]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(pivot_rate, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[0], cbar_kws={'label': '%'})
axes[0].set_title('Overstay Rate (%) by UCLA Quarter & Year')

sns.heatmap(pivot_sess, annot=True, fmt=',d', cmap='Blues', ax=axes[1], cbar_kws={'label': 'sessions'})
axes[1].set_title('Total Enforced Sessions by UCLA Quarter & Year')

plt.tight_layout()
plt.savefig(DATA_DIR + 'fig_quarter_year_heatmap.png', dpi=150)
plt.show()

In [ ]:
# 7c. Weekday vs Weekend
wd = enforced.groupby(['IsWeekend', 'DayName']).agg(
    Sessions=('IsOverstay', 'count'),
    Overstays=('IsOverstay', 'sum'),
    MeanOccMin=('OccupancyMin', 'mean'),
).assign(OverstayRate=lambda d: d['Overstays'] / d['Sessions'] * 100).reset_index()

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
wd['DayName'] = pd.Categorical(wd['DayName'], categories=day_order, ordered=True)
wd = wd.sort_values('DayName')

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4C72B0' if not w else '#DD8452' for w in wd['IsWeekend']]
ax.bar(wd['DayName'], wd['OverstayRate'], color=colors)
ax.set_title('Overstay Rate by Day of Week')
ax.set_ylabel('Overstay Rate (%)')
ax.set_xlabel('')
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#4C72B0', label='Weekday'),
    Patch(color='#DD8452', label='Weekend')
])
plt.tight_layout()
plt.savefig(DATA_DIR + 'fig_overstay_by_day.png', dpi=150)
plt.show()

print(wd[['DayName','Sessions','Overstays','OverstayRate','MeanOccMin']].to_string(index=False))

In [ ]:
# 7d. Time of day
tod = enforced.groupby('TimeOfDay', observed=True).agg(
    Sessions=('IsOverstay', 'count'),
    Overstays=('IsOverstay', 'sum'),
    MeanOccMin=('OccupancyMin', 'mean'),
).assign(OverstayRate=lambda d: d['Overstays'] / d['Sessions'] * 100)

fig, ax = plt.subplots(figsize=(9, 5))
tod['OverstayRate'].plot.bar(ax=ax, color='steelblue')
ax.set_title('Overstay Rate by Time of Day')
ax.set_ylabel('Overstay Rate (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig(DATA_DIR + 'fig_overstay_by_tod.png', dpi=150)
plt.show()

print(tod[['Sessions','Overstays','OverstayRate','MeanOccMin']].round(2).to_string())

In [ ]:
# 7e. Monthly time-series trend (overstay rate)
monthly = enforced.groupby(['Year', 'Month']).agg(
    Sessions=('IsOverstay', 'count'),
    Overstays=('IsOverstay', 'sum'),
).assign(OverstayRate=lambda d: d['Overstays'] / d['Sessions'] * 100).reset_index()
monthly['Date'] = pd.to_datetime(monthly[['Year', 'Month']].assign(day=1))
monthly = monthly.sort_values('Date')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['Date'], monthly['OverstayRate'], marker='o', linewidth=1.5)
ax.set_title('Monthly Overstay Rate — Westwood Curbside Parking')
ax.set_ylabel('Overstay Rate (%)')
ax.set_xlabel('')
ax.xaxis.set_major_locator(mticker.MaxNLocator(12))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(DATA_DIR + 'fig_monthly_trend.png', dpi=150)
plt.show()

In [ ]:
# 7f. Quarter × Weekday/Weekend heatmap (overstay rate)
qw = enforced.groupby(['Quarter', 'IsWeekend']).agg(
    Sessions=('IsOverstay', 'count'),
    Overstays=('IsOverstay', 'sum'),
).assign(OverstayRate=lambda d: d['Overstays'] / d['Sessions'] * 100).reset_index()
qw['WeekType'] = qw['IsWeekend'].map({True: 'Weekend', False: 'Weekday'})

pivot_qw = qw.pivot(index='Quarter', columns='WeekType', values='OverstayRate').loc[quarter_order]

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(pivot_qw, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, cbar_kws={'label': '%'})
ax.set_title('Overstay Rate (%) by Quarter & Weekday/Weekend')
plt.tight_layout()
plt.savefig(DATA_DIR + 'fig_quarter_weektype_heatmap.png', dpi=150)
plt.show()

---
## 8. Spatial / Hotspot Analysis

In [ ]:
# 8a. Overstay rate & mean overstay duration by BlockFace
hotspot = enforced.groupby('BlockFace').agg(
    Sessions=('IsOverstay', 'count'),
    Overstays=('IsOverstay', 'sum'),
    MeanOverstayMin=('OverstayMin', lambda x: x[x > 0].mean()),
    Lat=('Latitude', 'first'),
    Lon=('Longitude', 'first'),
).assign(OverstayRate=lambda d: d['Overstays'] / d['Sessions'] * 100).reset_index()

hotspot_top = hotspot.sort_values('OverstayRate', ascending=False).head(15)
print('Top 15 overstay BlockFaces:')
print(hotspot_top[['BlockFace','Sessions','Overstays','OverstayRate','MeanOverstayMin']].round(1).to_string(index=False))

In [ ]:
# 8b. Bar chart — top 20 hotspots
top20 = hotspot.sort_values('OverstayRate', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top20['BlockFace'][::-1], top20['OverstayRate'][::-1], color='tomato')
ax.set_title('Top 20 Overstay Hotspots by BlockFace')
ax.set_xlabel('Overstay Rate (%)')
plt.tight_layout()
plt.savefig(DATA_DIR + 'fig_hotspot_bar.png', dpi=150)
plt.show()

In [ ]:
# 8c. Bubble map — spatial distribution of overstay rate
# Bubble size = number of overstay sessions, color = overstay rate
fig, ax = plt.subplots(figsize=(10, 8))

sc = ax.scatter(
    hotspot['Lon'], hotspot['Lat'],
    s=hotspot['Overstays'] / hotspot['Overstays'].max() * 400 + 20,
    c=hotspot['OverstayRate'],
    cmap='YlOrRd', alpha=0.75, edgecolors='grey', linewidths=0.4
)
plt.colorbar(sc, ax=ax, label='Overstay Rate (%)')
ax.set_title('Westwood Parking Overstay Hotspot Map\n(bubble size ∝ overstay count)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig(DATA_DIR + 'fig_hotspot_map.png', dpi=150)
plt.show()

In [ ]:
# 8d. Hotspot × Quarter: does spatial pattern shift seasonally?
spot_quarter = enforced.groupby(['BlockFace', 'Quarter']).agg(
    Sessions=('IsOverstay', 'count'),
    Overstays=('IsOverstay', 'sum'),
).assign(OverstayRate=lambda d: d['Overstays'] / d['Sessions'] * 100).reset_index()

# Pivot: top 15 blocks × 4 quarters
top15_blocks = hotspot.sort_values('OverstayRate', ascending=False).head(15)['BlockFace'].tolist()
pivot_sq = spot_quarter[spot_quarter['BlockFace'].isin(top15_blocks)].pivot(
    index='BlockFace', columns='Quarter', values='OverstayRate'
)[quarter_order]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot_sq, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, cbar_kws={'label': '%'})
ax.set_title('Overstay Rate (%) — Top 15 Blocks × UCLA Quarter')
plt.tight_layout()
plt.savefig(DATA_DIR + 'fig_hotspot_quarter.png', dpi=150)
plt.show()

---
## 9. ML — Overstay Prediction

Binary classification: **will this session overstay?**  
Features: temporal (quarter, weekday, hour), spatial (BlockFace), time limit, session context.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay
)
import shap

In [ ]:
ml_df = enforced[[
    'IsOverstay', 'Quarter', 'IsWeekend', 'DayOfWeek',
    'StartHour', 'TimeLimitMinutes', 'BlockFace', 'Year'
]].dropna(subset=['TimeLimitMinutes']).copy()

# Encode categoricals
le_q  = LabelEncoder(); ml_df['Quarter_enc']   = le_q.fit_transform(ml_df['Quarter'])
le_bf = LabelEncoder(); ml_df['BlockFace_enc'] = le_bf.fit_transform(ml_df['BlockFace'])

FEATURES = ['Quarter_enc', 'IsWeekend', 'DayOfWeek', 'StartHour', 'TimeLimitMinutes', 'BlockFace_enc', 'Year']
TARGET   = 'IsOverstay'

X = ml_df[FEATURES].values
y = ml_df[TARGET].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')
print(f'Overstay prevalence (train): {y_train.mean()*100:.1f}%')

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, y_pred, target_names=['No Overstay', 'Overstay']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}')

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, label=f'Random Forest (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Overstay Prediction')
ax.legend()
plt.tight_layout()
plt.savefig(DATA_DIR + 'fig_roc.png', dpi=150)
plt.show()

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['No Overstay','Overstay'], ax=ax)
ax.set_title('Confusion Matrix — Random Forest')
plt.tight_layout()
plt.savefig(DATA_DIR + 'fig_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Feature importance
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
fi.plot.barh(ax=ax, color='steelblue')
ax.set_title('Feature Importance — Random Forest')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(DATA_DIR + 'fig_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# SHAP values (sample 5000 for speed)
sample_idx = np.random.choice(len(X_test), size=min(5000, len(X_test)), replace=False)
explainer  = shap.TreeExplainer(rf)
shap_vals  = explainer.shap_values(X_test[sample_idx])

shap.summary_plot(
    shap_vals[1], X_test[sample_idx],
    feature_names=FEATURES, show=False
)
plt.tight_layout()
plt.savefig(DATA_DIR + 'fig_shap_summary.png', dpi=150)
plt.show()

---
## 10. Summary table for paper (Table 1)

Overstay statistics by Year × Quarter × WeekType — ready to paste into the paper.

In [ ]:
summary_table = enforced.groupby(['Year', 'Quarter', 'IsWeekend']).agg(
    N_Sessions=('IsOverstay', 'count'),
    N_Overstay=('IsOverstay', 'sum'),
    OverstayRate_pct=('IsOverstay', lambda x: x.mean() * 100),
    Median_OccMin=('OccupancyMin', 'median'),
    Mean_OverstayMin=('OverstayMin', lambda x: x[x > 0].mean()),
).reset_index()

summary_table['WeekType'] = summary_table['IsWeekend'].map({True: 'Weekend', False: 'Weekday'})
summary_table = summary_table.drop(columns='IsWeekend')
summary_table['Quarter'] = pd.Categorical(summary_table['Quarter'], categories=quarter_order, ordered=True)
summary_table = summary_table.sort_values(['Year', 'Quarter', 'WeekType'])

print(summary_table.round(2).to_string(index=False))
summary_table.round(2).to_csv(DATA_DIR + 'table1_summary.csv', index=False)
print('\nSaved → table1_summary.csv')